# 서울 아파트 실거래가 수집기
**마포구 / 용산구 / 성동구 · 2020~2025년**

순서대로 셀을 실행하면 됩니다. ▶ 버튼을 위에서부터 차례로 누르세요.

## 1단계 · 저장소 클론 및 패키지 설치

In [ ]:
# 저장소 클론
!git clone https://github.com/jiuk96/realestate_analysis_tool.git
%cd realestate_analysis_tool
!pip install -q -r requirements.txt
print('✅ 설치 완료')

## 2단계 · API 키 / GitHub 토큰 입력

> 아래 칸에 키를 입력하고 ▶ 를 누르세요. 키는 Colab 세션 메모리에만 저장되며 외부에 노출되지 않습니다.

In [ ]:
import os
from getpass import getpass

molit_key = getpass('국토교통부 API 키 입력: ')
git_token = getpass('GitHub Token 입력 (repo 권한): ')

# .env 파일에 저장
with open('.env', 'w') as f:
    f.write(f'MOLIT_API_KEY={molit_key}\n')
    f.write(f'GIT_TOKEN={git_token}\n')

print('✅ 키 설정 완료 (.env 저장됨)')

## 3단계 · API 연결 테스트 (수집 전 확인)

In [ ]:
import os, requests
from dotenv import load_dotenv
load_dotenv()

key = os.getenv('MOLIT_API_KEY')
resp = requests.get(
    'http://apis.data.go.kr/1613000/RTMSDataSvcAptTrade/getRTMSDataSvcAptTrade',
    params={'serviceKey': key, 'LAWD_CD': '11440', 'DEAL_YMD': '202001',
            'pageNo': 1, 'numOfRows': 3, 'resultType': 'json'},
    timeout=15
)
print(f'HTTP 상태코드: {resp.status_code}')

if resp.status_code == 200:
    data = resp.json()
    total = data['response']['body']['totalCount']
    print(f'✅ API 연결 성공! 마포구 2020년 1월 거래 건수: {total}건')
else:
    print(f'❌ 연결 실패: {resp.text[:300]}')

## 4단계 · 데이터 수집 실행

마포구 / 용산구 / 성동구 × 72개월 = 약 216회 API 호출

**예상 소요 시간: 10~20분**

수집 중 중단되어도 체크포인트 덕분에 재실행 시 이어서 진행됩니다.

In [ ]:
import sys
sys.path.insert(0, '.')

from src.collector import collect_all
df = collect_all()

if not df.empty:
    print(f'\n✅ 수집 완료!')
    print(f'   총 거래 건수 : {len(df):,}건')
    print(f'   단지 수      : {df["apt_name"].nunique()}개')
    print(f'   메모리 사용량: {df.memory_usage(deep=True).sum()/1024**2:.1f} MB')
    print()
    print('구별 거래 건수:')
    print(df.groupby('district_name', observed=True)['deal_amount'].count().to_string())

## 5단계 · GitHub에 푸시

수집된 데이터를 GitHub에 저장합니다. 이후 어떤 환경에서도 `git pull` 한 번으로 데이터를 불러올 수 있습니다.

In [ ]:
from src.collector import git_push_data

# git 사용자 정보 설정 (Colab 환경에 git config 없을 수 있음)
!git config user.email 'jiuk9612@gmail.com'
!git config user.name 'jiuk96'

success = git_push_data('data: 마용성 실거래가 수집 완료 (2020~2025)')

if success:
    print('✅ GitHub 푸시 완료!')
    print('   → https://github.com/jiuk96/realestate_analysis_tool')
    print('   이제 Claude Code 환경에서 git pull 후 분석을 이어할 수 있습니다.')
else:
    print('❌ 푸시 실패 — GitHub Token 권한(repo)을 확인해주세요.')

## 수집 완료 후 다음 단계

Claude Code 환경에서 아래 명령어를 실행하면 실거래 데이터로 분석이 바로 시작됩니다:

```bash
git pull
python web/app.py
# → http://localhost:5000
```